
# Silver Pedidos

**Fuente:** `prod_dataco.brz_dataco.raw_dataco_supply_chain` (histórico completo acumulado)

**Destino:** Tabla de **Pedidos**, en `prod_dataco.slv_dataco`

**Grano:** un registro por `order_id`

## Qué hace

1. Lee todo el histórico de Bronze acumulado, no solo el último corte.
2. Deduplica por `order_id`, priorizando el registro correspondiente al corte más reciente mediante `dedup_reciente`, ordenando por `fecha_fin_corte` descendente.
3. Filtra `order_id` nulo, vacío o `'0'`, equivalente a la regla del SP original.
4. Parsea las fechas utilizando las columnas string originales:

   * `order_date_dateorders`
   * `shipping_date_dateorders`

   utilizando los formatos de fecha definidos en el notebook, preservando la hora cuando está disponible.
5. Limpia y normaliza los datos:

   * Campos de texto: `trim` + mayúsculas.
   * `order_customer_id`: `int`.
   * `late_delivery_risk`: `int`.
   * `days_for_shipping_real`: `int`.
   * `days_for_shipment_scheduled`: `int`.
6. Conserva las columnas de trazabilidad provenientes de Bronze.
7. Escribe la tabla Silver mediante `overwrite` completo.

## Columnas de negocio

* `order_id`
* `order_customer_id`
* `order_date`
* `shipping_date`
* `order_status`
* `delivery_status`
* `late_delivery_risk`
* `days_for_shipping_real`
* `days_for_shipment_scheduled`
* `shipping_mode`
* `market`
* `order_region`
* `order_country`
* `order_state`
* `order_city`
* `order_zipcode`

## Trazabilidad conservada

* `fecha_inicio_corte`
* `fecha_fin_corte`
* `fecha_carga`
* `id_ejecucion`

## Nota de diseño

Las fechas de pedido y envío se obtienen desde las columnas string originales de Bronze para evitar perder la hora debido a la inferencia de tipos realizada durante la ingesta.

La tabla utiliza `dedup_reciente` para priorizar el registro correspondiente al corte más reciente cuando existen múltiples registros para un mismo `order_id`.

## Por qué importa este notebook

La tabla de **Pedidos** representa la cabecera de los pedidos y constituye una de las principales fuentes para los indicadores de Gold.

Los filtros de negocio específicos para cada indicador, incluyendo la exclusión de pedidos cancelados, se aplican en la capa Gold y no en esta tabla Silver.

In [0]:
# ============================================================
# Celda 1: Parámetros + lectura de Bronze + helpers
# ------------------------------------------------------------
# Mismo patrón que las 3 dimensiones. Notebook con sesión propia,
# por eso repite todo desde cero.
# ============================================================

dbutils.widgets.text("bronze_catalog", "prod_dataco", "Catálogo Bronze")
dbutils.widgets.text("bronze_schema", "brz_dataco", "Esquema Bronze")
dbutils.widgets.text("bronze_table", "raw_dataco_supply_chain", "Tabla Bronze")

dbutils.widgets.text("silver_catalog", "prod_dataco", "Catálogo Silver")
dbutils.widgets.text("silver_schema", "slv_dataco", "Esquema Silver")

bronze_catalog = dbutils.widgets.get("bronze_catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
bronze_table = dbutils.widgets.get("bronze_table")

silver_catalog = dbutils.widgets.get("silver_catalog")
silver_schema = dbutils.widgets.get("silver_schema")

BRONZE_TABLE_FQN = f"{bronze_catalog}.{bronze_schema}.{bronze_table}"
SILVER_PEDIDOS_FQN = f"{silver_catalog}.{silver_schema}.hd_pedidos"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{silver_catalog}.{silver_schema}"
)

df_bronze = spark.table(BRONZE_TABLE_FQN)

from pyspark.sql import functions as F, Window

def clean_str(col_name: str, default: str = "") -> F.Column:
    return F.upper(F.trim(F.coalesce(F.col(col_name).cast("string"), F.lit(default))))

def clean_num(col_name: str, cast_type, default: str = "0") -> F.Column:
    trimmed = F.trim(F.col(col_name).cast("string"))
    nulled = F.when((trimmed.isNull()) | (trimmed == ""), None).otherwise(trimmed)
    return F.coalesce(nulled, F.lit(default)).cast(cast_type)

def dedup_reciente(df, partition_cols: list, orden_col: str):
    w = Window.partitionBy(*partition_cols).orderBy(F.col(orden_col).desc())
    return (
        df.withColumn("rn", F.row_number().over(w))
          .filter(F.col("rn") == 1)
          .drop("rn")
    )

print(f"Total filas en Bronze: {df_bronze.count()}")
print("✓ Helpers definidos: clean_str, clean_num, dedup_reciente")

In [0]:
# ============================================================
# INICIO DE LOG DE EJECUCIÓN — slv_orders
# ============================================================

import uuid

id_ejecucion = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control",
    key="id_ejecucion",
    debugValue=""
)

LOG_ID = str(uuid.uuid4())
NOTEBOOK = "slv_orders"
CAPA = "silver"

spark.sql(f"""
INSERT INTO prod_dataco.metadata.etl_log
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)

print(f"✓ Registro EN_PROCESO creado en etl_log: {LOG_ID}")

In [0]:
[c for c in df_bronze.columns if any(k in c.lower() for k in ["order_id", "order_customer_id", "order_status", "delivery_status", "days_for_ship", "shipping_mode", "date", "late_delivery"])]

In [0]:
df_bronze.select("order_date_dateorders", "order_date", "shipping_date_dateorders").printSchema()
df_bronze.select("order_date_dateorders", "order_date", "shipping_date_dateorders").show(5, truncate=False)

In [0]:
# ============================================================
# Celda 2: Parseo de fechas + Deduplicación + construcción de hd_pedidos
# ------------------------------------------------------------
# Fechas: uso order_date_dateorders / shipping_date_dateorders (string,
# formato "M/d/yyyy H:mm" confirmado con muestra real) en vez de la
# columna 'order_date' ya tipada, porque esta última perdió la hora
# al inferir schema y no hay 'shipping_date' equivalente -> por
# consistencia, parseo ambas desde el string original.
#
# dedup: por order_id, determinístico (fecha_fin_corte desc), igual
# criterio que md_ubicaciones.
#
# Filtro: excluye Order Id nulo/vacío/'0' -> igual regla que el SP
# original (WHERE rn=1 AND [Order Id] IS NOT NULL AND TRIM(...) <> '0')
#
# Se conservan las columnas de trazabilidad provenientes de Bronze
# para mantener la trazabilidad de extremo a extremo.
# ============================================================

FECHA_FORMATOS = ["M/d/yyyy H:mm", "MM/dd/yyyy HH:mm:ss", "MM/dd/yyyy"]

def clean_date(col_name: str) -> F.Column:
    trimmed = F.trim(F.col(col_name).cast("string"))
    resultado = F.lit(None).cast("timestamp")
    for fmt in FECHA_FORMATOS:
        resultado = F.coalesce(resultado, F.to_timestamp(trimmed, fmt))
    return resultado

df_pedidos_dedup = dedup_reciente(
    df_bronze,
    ["order_id"],
    orden_col="fecha_fin_corte"
)

df_pedidos_dedup = df_pedidos_dedup.filter(
    F.col("order_id").isNotNull() &
    (F.trim(F.col("order_id").cast("string")) != "") &
    (F.trim(F.col("order_id").cast("string")) != "0")
)

df_silver_pedidos = df_pedidos_dedup.select(
    clean_num("order_id", "int").alias("order_id"),
    clean_num("order_customer_id", "int").alias("order_customer_id"),
    clean_date("order_date_dateorders").alias("order_date"),
    clean_date("shipping_date_dateorders").alias("shipping_date"),
    clean_str("order_status").alias("order_status"),
    clean_str("delivery_status").alias("delivery_status"),
    clean_num("late_delivery_risk", "int").alias("late_delivery_risk"),
    clean_num("days_for_shipping_real", "int").alias("days_for_shipping_real"),
    clean_num("days_for_shipment_scheduled", "int").alias("days_for_shipment_scheduled"),
    clean_str("shipping_mode").alias("shipping_mode"),
    clean_str("market").alias("market"),
    clean_str("order_region").alias("order_region"),
    clean_str("order_country").alias("order_country"),
    clean_str("order_state").alias("order_state"),
    clean_str("order_city").alias("order_city"),
    clean_str("order_zipcode").alias("order_zipcode"),

    # --------------------------------------------------------
    # Trazabilidad
    # --------------------------------------------------------
    F.col("fecha_inicio_corte"),
    F.col("fecha_fin_corte"),
    F.col("fecha_carga"),
    F.col("id_ejecucion")
)

print(f"Filas antes del dedup/filtro : {df_bronze.count()}")
print(f"Filas después                : {df_silver_pedidos.count()}")

display(df_silver_pedidos.limit(10))

In [0]:
# ============================================================
# Celda 3: Escritura a silver_pedidos
# ------------------------------------------------------------
# overwrite completo, mismo patrón que las 3 dimensiones.
# ============================================================

(
    df_silver_pedidos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_PEDIDOS_FQN)
)

print(f"✓ {df_silver_pedidos.count()} filas escritas en {SILVER_PEDIDOS_FQN}")

In [0]:
# ============================================================
# ACTUALIZACIÓN DEL LOG — EJECUCIÓN EXITOSA
# ============================================================

spark.sql(f"""
UPDATE prod_dataco.metadata.etl_log
SET estado = 'EXITOSO', fecha_fin = current_timestamp()
WHERE log_id = '{LOG_ID}'
""")

print("✓ etl_log actualizado: EXITOSO")